# Lane 2: Diff-JEPA - Google Colab Runner
**Project:** Action-Asymmetric Differential Isotropic JEPA  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Does subtracting an unconditioned passive background head from an action-conditioned causal head with SIGReg isolate causal affordances under heavy visual clutter?

Configured for Google Colab (Free T4 or Pro) and local GPUs.


In [ ]:
# Step 1: Pre-flight Hardware Verification & Credit Guard
import sys, os, time, math
import torch
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Unassigning Colab GPU immediately to prevent idle burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Device: {dev_name} | VRAM: {vram_gb:.2f} GB")
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 2: Dependencies & Standalone Engine (Dual-mode)
try:
    from diff_jepa.models import ActionAsymmetricDualPredictor, VisualStateEncoder
    from diff_jepa.sigreg import SIGRegLoss
    from diff_jepa.metrics import compute_effective_rank
    print("Loaded Diff-JEPA from installed package.")
except ImportError:
    print("Package not found on path; loading self-contained standalone implementation...")
    import torch.nn as nn
    import torch.nn.functional as F
    
    class VisualStateEncoder(nn.Module):
        def __init__(self, d_raw: int, d_state: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d_raw, d_state * 2),
                nn.LayerNorm(d_state * 2),
                nn.GELU(),
                nn.Linear(d_state * 2, d_state),
                nn.LayerNorm(d_state)
            )
        def forward(self, x): return self.net(x)
    
    class ActionAsymmetricDualPredictor(nn.Module):
        def __init__(self, d_state: int, d_action: int, d_out: int, lambda_min: float = 0.20, lambda_max: float = 0.85):
            super().__init__()
            self.causal_head = nn.Sequential(nn.Linear(d_state + d_action, d_out), nn.GELU(), nn.Linear(d_out, d_out))
            self.passive_head = nn.Sequential(nn.Linear(d_state + d_action, d_out), nn.GELU(), nn.Linear(d_out, d_out))
            self.raw_lambda = nn.Parameter(torch.tensor(1.0))
            self.lambda_min, self.lambda_max = lambda_min, lambda_max
        def get_lambda(self):
            return self.lambda_min + (self.lambda_max - self.lambda_min) * torch.sigmoid(self.raw_lambda)
        def forward(self, state, action):
            z1 = self.causal_head(torch.cat([state, action], dim=-1))
            z2 = self.passive_head(torch.cat([state, torch.zeros_like(action)], dim=-1))
            diff = z1 - self.get_lambda() * z2
            diff_mean = diff.mean(dim=0, keepdim=True)
            diff_std = diff.std(dim=0, keepdim=True) + 1e-5
            return (diff - diff_mean) / diff_std
    
    class SIGRegLoss(nn.Module):
        def __init__(self, d: int, n_slices: int = 32):
            super().__init__()
            q, _ = torch.linalg.qr(torch.randn(d, n_slices))
            self.register_buffer('proj', q)
            t_vals = torch.linspace(0.1, 2.0, 10)
            self.register_buffer('t_vals', t_vals)
            self.register_buffer('exp_std', torch.exp(-0.5 * (t_vals ** 2)))
        def forward(self, z):
            sketched = torch.mm(z, self.proj)
            cos_terms = torch.cos(sketched.unsqueeze(-1) * self.t_vals)
            return (cos_terms.mean(dim=0) - self.exp_std).pow(2).mean()
    
    def compute_effective_rank(Z: torch.Tensor) -> float:
        with torch.no_grad():
            Z_c = Z.float() - Z.float().mean(dim=0, keepdim=True)
            S = torch.linalg.svdvals(Z_c)
            tot = torch.sum(S)
            if tot == 0: return 0.0
            p = S / tot
            return math.exp(-torch.sum(p * torch.log(p + 1e-12)).item())
    print("Standalone Diff-JEPA initialized.")


In [ ]:
# Step 3: Run Push-T Affordance Smoke Test under 80% Clutter
import torch.nn.functional as F

torch.manual_seed(2026)
d_causal, d_action, d_noise = 16, 4, 112
d_total = d_causal + d_noise
batch_size, n_steps = 64, 300

encoder = VisualStateEncoder(d_raw=d_total, d_state=d_total).to(device)
predictor = ActionAsymmetricDualPredictor(d_state=d_total, d_action=d_action, d_out=d_total).to(device)
sigreg_diff = SIGRegLoss(d=d_total, n_slices=32).to(device)
sigreg_enc = SIGRegLoss(d=d_total, n_slices=32).to(device)

opt = torch.optim.Adam(list(encoder.parameters()) + list(predictor.parameters()), lr=1e-3)

t0 = time.time()
losses = []
latents = []
W_action = torch.randn(d_action, d_causal, device=device) * 0.5

for step in range(n_steps):
    s_causal = torch.randn(batch_size, d_causal, device=device)
    action = torch.randn(batch_size, d_action, device=device)
    s_next_causal = 1.2 * s_causal + action @ W_action
    
    bg_shared = 3.0 * torch.randn(batch_size, d_noise, device=device)
    raw_curr = torch.cat([s_causal, bg_shared + 0.2*torch.randn(batch_size, d_noise, device=device)], dim=-1)
    raw_next = torch.cat([s_next_causal, bg_shared + 0.2*torch.randn(batch_size, d_noise, device=device)], dim=-1)
    
    s_curr = encoder(raw_curr)
    with torch.no_grad():
        s_target = encoder(raw_next)
        
    pred_diff = predictor(s_curr, action)
    target_norm = (s_target - s_target.mean(dim=0, keepdim=True)) / (s_target.std(dim=0, keepdim=True) + 1e-5)
    
    l_pred = F.mse_loss(pred_diff, target_norm)
    l_sigreg_diff = sigreg_diff(pred_diff)
    l_sigreg_enc = sigreg_enc(s_curr)
    loss = l_pred + 0.1 * l_sigreg_diff + 0.05 * l_sigreg_enc
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    
    losses.append(loss.item())
    if step >= n_steps - 50:
        latents.append(pred_diff.detach())
    if step % 50 == 0 or step == n_steps - 1:
        print(f"Step {step:03d} | Loss: {loss.item():.4f} | Lambda: {predictor.get_lambda().item():.3f}")

elapsed = time.time() - t0
final_latents = torch.cat(latents, dim=0)
erank = compute_effective_rank(final_latents)
erank_ratio = erank / d_total
print(f"\nCompleted in {elapsed:.2f}s | Effective Rank: {erank:.2f} / {d_total} (Ratio: {erank_ratio:.2f})")


In [ ]:
# Step 4: Pre-Registered Gate Assessment
passed = erank_ratio >= 0.70 and losses[-1] < losses[0]
print("=" * 60)
print(f"Effective Rank Ratio: {erank_ratio:.2f} (Target: >= 0.70)")
print(f"Initial Loss: {losses[0]:.4f} -> Final Loss: {losses[-1]:.4f}")
print(f"Pre-Registered Gate: {'PASSED (No collapse & loss decreased)' if passed else 'FAILED (Kill)'}")
print("=" * 60)


In [ ]:
# Step 5: Credit Protection & Automatic Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
